### **Gold Layer**

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

In [0]:
from pyspark.sql.functions import (col, to_date, date_format, year, month,
                                    dayofmonth, dayofweek, sum as _sum, count)

dim_date = (spark.sql(
        "SELECT explode(sequence(to_date('2024-09-01'), to_date('2024-11-30'), interval 1 day)) AS date")
    .withColumn("date_key",    date_format("date", "yyyyMMdd").cast("int"))   # smart integer key: 20241003
    .withColumn("year",        year("date"))
    .withColumn("month",       month("date"))
    .withColumn("month_name",  date_format("date", "MMMM"))
    .withColumn("day",         dayofmonth("date"))
    .withColumn("day_of_week", date_format("date", "EEEE"))
    .withColumn("is_weekend",  dayofweek("date").isin(1, 7)))

dim_date.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_date")   # 91 rows

In [0]:
(spark.table("workspace.silver.customers")
    .select("account_id","first_name","last_name","region","segment","tariff_id","has_email","signup_date")
    .write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_customer"))   # 50

(spark.table("workspace.silver.meters")
    .select("meter_id","account_id","meter_type","supply_id","install_date")
    .write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_meter"))      # 59

(spark.table("workspace.silver.tariffs")
    .write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_tariff"))     # 4

In [0]:
meter_acct = spark.table("workspace.gold.dim_meter").select("meter_id", "account_id")

fact_consumption = (spark.table("workspace.silver.readings")
    .withColumn("date", to_date("read_timestamp"))
    .groupBy("meter_id", "date")                      # <-- this groupBy IS the grain: meter × day
    .agg(_sum("consumption_kwh").alias("total_kwh"),
         count("*").alias("reading_count"),
         _sum(col("is_estimated").cast("int")).alias("estimated_count"),
         _sum(col("is_implausible").cast("int")).alias("implausible_count"))
    .withColumn("date_key", date_format("date", "yyyyMMdd").cast("int"))   # link to dim_date
    .join(meter_acct, "meter_id", "left"))                                 # carry account_id -> link to dim_customer

fact_consumption.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.fact_consumption")
print(spark.table("workspace.gold.fact_consumption").count(), "rows")      # 1,791

In [0]:
fact_billing = (spark.table("workspace.silver.invoices")
    .withColumn("date_key", date_format("issued_date", "yyyyMMdd").cast("int"))
    .select("invoice_id","account_id","date_key","period_start","period_end",
            "total_kwh","amount_pence","status"))
fact_billing.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.fact_billing")

In [0]:
display(
    spark.table("workspace.gold.fact_consumption").alias("f")
        .join(spark.table("workspace.gold.dim_customer").alias("c"), "account_id")
        .groupBy("region")
        .agg(_sum("total_kwh").alias("total_kwh"))
        .orderBy(col("total_kwh").desc()))

In [0]:
from pyspark.sql.functions import xxhash64, concat_ws, col, to_date, lit, date_sub, when

dim_meter = (spark.table("workspace.silver.meters")
    .select("meter_id","account_id","meter_type","supply_id","install_date")
    .withColumn("meter_key", xxhash64("meter_id")))
dim_meter.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.dim_meter")

dim_tariff = (spark.table("workspace.silver.tariffs")
    .withColumn("tariff_key", xxhash64("tariff_id")))
dim_tariff.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.dim_tariff")

In [0]:
# For slowly changing dimensions SCD2
INIT_DATE = "2024-09-01"

dim_customer = (spark.table("workspace.silver.customers")
    .select("account_id","first_name","last_name","region","segment","tariff_id","has_email")
    .withColumn("valid_from", to_date(lit(INIT_DATE)))
    .withColumn("valid_to",   to_date(lit("9999-12-31")))   # sentinel = this version is still open
    .withColumn("is_current", lit(True))
    .withColumn("customer_key", xxhash64(concat_ws("|", col("account_id"), col("valid_from")))))

dim_customer.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.dim_customer")
print(dim_customer.count(), "customers, all current")   # 50

In [0]:
# Example of a change in customer's region
CHANGE_DATE = "2024-10-15"
mover = "ACC-100016"

dim = spark.table("workspace.gold.dim_customer")

# 1) CLOSE the mover's current version
closed = (dim
    .withColumn("valid_to",   when(col("account_id")==mover, date_sub(to_date(lit(CHANGE_DATE)), 1)).otherwise(col("valid_to")))
    .withColumn("is_current", when(col("account_id")==mover, lit(False)).otherwise(col("is_current"))))

# 2) INSERT the new version — new region, new key, open-ended
new_version = (dim.filter(col("account_id")==mover)
    .withColumn("region", lit("London"))
    .withColumn("valid_from", to_date(lit(CHANGE_DATE)))
    .withColumn("valid_to",   to_date(lit("9999-12-31")))
    .withColumn("is_current", lit(True))
    .withColumn("customer_key", xxhash64(concat_ws("|", col("account_id"), to_date(lit(CHANGE_DATE))))))

dim_customer = closed.unionByName(new_version)
dim_customer.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_customer")

dim_customer.filter(col("account_id")==mover).select(
    "customer_key","account_id","region","valid_from","valid_to","is_current").orderBy("valid_from").show(truncate=False)

In [0]:
from pyspark.sql.functions import to_date, date_format, sum as _sum, count, col

dimc       = spark.table("workspace.gold.dim_customer").select("customer_key","account_id","valid_from","valid_to")
meter_acct = spark.table("workspace.gold.dim_meter").select("meter_id","account_id")

fact_consumption = (spark.table("workspace.silver.readings")
    .withColumn("date", to_date("read_timestamp"))
    .groupBy("meter_id", "date")
    .agg(_sum("consumption_kwh").alias("total_kwh"),
         count("*").alias("reading_count"),
         _sum(col("is_estimated").cast("int")).alias("estimated_count"),
         _sum(col("is_implausible").cast("int")).alias("implausible_count"))
    .join(meter_acct, "meter_id", "left")
    .withColumn("date_key", date_format("date", "yyyyMMdd").cast("int")))

# attach the customer_key of the version that was current ON THE EVENT DATE
fact_consumption = (fact_consumption.alias("f")
    .join(dimc.alias("d"),
          (col("f.account_id") == col("d.account_id")) &
          (col("f.date").between(col("d.valid_from"), col("d.valid_to"))),   # <-- the temporal match
          "left")
    .select("f.*", "d.customer_key"))

fact_consumption.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.fact_consumption")